<a href="https://colab.research.google.com/github/RQfreefly/notebooks/blob/main/finetuning/notebooks/lfm3_cpt_with_trl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 使用 TRL 对 LFM3-1.7B 进行持续预训练 (CPT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/lfm3_cpt_with_trl.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-Liquid4All%2Fcookbook-181717?logo=github)](https://github.com/Liquid4All/cookbook) [![Discord](https://img.shields.io/discord/1385439864920739850?color=7289da&label=Discord&logo=discord&logoColor=white)](https://discord.com/invite/liquid-ai)

本笔记本展示了如何使用 [TRL](https://github.com/huggingface/trl) 和 LoRA，在来自新领域的原始文本上对 [LFM3-1.7B](https://huggingface.co/LiquidAI/LFM3-1.7B) 进行持续预训练（Continued Pre-Training）。它可以在免费的 Colab T4 GPU 或任何显存至少为 16 GB 的 GPU 上运行。

持续预训练 (CPT) 通过无标注文本，使用与预训练相同的下一 token 预测目标，向模型传授特定领域的词汇、风格和知识。在本教程中，我们的目标领域是**土耳其语**：LFM3-1.7B 官方支持 16 种语言，而土耳其语并不在其中。我们将在土耳其语维基百科文章上进行训练，并衡量模型在训练前后对保留的土耳其语文本的预测能力。

## 🎯 您将学到什么

- **原始文本数据**：从 Hugging Face Hub 流式传输大型语料库的分片
- **LoRA + CPT**：在打包成完整行的纯文本上进行训练，就像在预训练中一样
- **评估**：在训练前后，评估模型在保留的维基百科文章和分布外网页上的困惑度（perplexity），以及进行土耳其语语法测试
- **导出**：将 LoRA 权重合并为一个独立的检查点

# 📦 安装与设置

In [1]:
!pip install -qqq "torch==2.14.1" "torchvision==0.29.1" "torchao==0.18.0" "trl==1.16.0" "transformers==5.20.2" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "huggingface-hub==2.3.0" "tokenizers==0.23.3" "safetensors==0.8.0" --progress-bar off

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
import math
import os
import torch
import transformers
import trl
import peft

os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"🎛️ PEFT version: {peft.__version__}")

# LFM3-1.7B 是以 bfloat16 发布的。不支持 bfloat16 的 GPU（例如 T4）会将其加载为 float16：
# 这种转换对于几乎每个权重都是精确的，而且无论哪种方式，LoRA 适配器都以 float32 进行训练。
use_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"🔢 Precision: {dtype}")

W1120 04:49:08.675000 2218 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1120 04:49:08.899000 2218 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


📦 PyTorch version: 2.14.1+cu130
🤗 Transformers version: 5.20.2
📊 TRL version: 1.16.0
🎛️ PEFT version: 0.21.2
🔢 Precision: torch.float16


# 从 Transformers 🤗 加载模型

我们继续训练经过微调的 LFM3-1.7B，这使其保留了对话能力，以便进行通常在 CPT 之后进行的指令微调。对于大规模的 CPT 任务，建议考虑从基础（Base）检查点开始训练。

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "LiquidAI/LFM3-1.7B"

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)

print("✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")

📚 Loading tokenizer...


config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_lfm3.py:   0%|          | 0.00/6.11k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/LFM3-1.7B:
- configuration_lfm3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/1.54k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.9MB            

... (8 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/286 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

✅ Model loaded successfully!
🔢 Parameters: 1,728,801,536


# 📚 加载领域语料库

[Wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia) 数据集每个语言都有一个子集，仅土耳其语一个子集就有超过 50 万篇文章。我们采用流式传输方式，这样就只需要下载我们将要使用的文章：3,000 篇用于训练，100 篇用于评估。为了验证模型学到的是土耳其语，而不仅仅是维基百科的写作风格，我们还在来自 [FineWeb-2](https://huggingface.co/datasets/HuggingFaceFW/fineweb-2) 的 100 个土耳其语网页（新闻、论坛、商店）上进行评估，这些网页模型在训练期间从未见过。

每篇文章都会转换成一个 `text` 示例，这是 TRL 进行纯语言建模所期望的格式。我们添加了结束符（end-of-sequence token），以便模型学习文档在哪里结束。

In [4]:
from datasets import Dataset, load_dataset

stream = load_dataset("wikimedia/wikipedia", "20231101.tr", split="train", streaming=True).shuffle(seed=42, buffer_size=10_000)

def to_text(article):
    return {"text": f"{article['title']}\n\n{article['text']}{tokenizer.eos_token}"}

# 跳过字数较少的草稿文章（通常是列表和模板）
articles = (to_text(article) for article in stream if len(article["text"]) > 2_000)
eval_dataset = Dataset.from_list([next(articles) for _ in range(100)])
train_dataset = Dataset.from_list([next(articles) for _ in range(3_000)])

# 分布外（OOD）检查：土耳其语网页（新闻、论坛、商店），与维基百科的文风不同
web = load_dataset("HuggingFaceFW/fineweb-2", name="tur_Latn", split="test", streaming=True)
ood_dataset = Dataset.from_list([{"text": page["text"] + tokenizer.eos_token} for page in web.take(100)])

n_tokens = sum(len(ids) for ids in tokenizer(list(train_dataset["text"]))["input_ids"])
print(f"📚 Train articles: {len(train_dataset)} ({n_tokens / 1e6:.1f}M tokens)")
print(f"🧪 Eval articles: {len(eval_dataset)}")
print(f"🌐 Out-of-distribution web pages: {len(ood_dataset)}")
print(f"\n📝 Sample:\n{train_dataset[0]['text'][:500]}")

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/329k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

📚 Train articles: 3000 (10.7M tokens)
🧪 Eval articles: 100
🌐 Out-of-distribution web pages: 100

📝 Sample:
Sıkıyönetim

Sıkıyönetim veya askerî adalet; askerî otoritenin, genellikle resmî bildirgesi altında, adli yönetimi kontrol altına almasıyla işleme geçen kural sistemidir.

Askerî adalet; ortaya çıkan savaş, doğal afet, sivil kargaşa, toprak işgali veya askerî darbe gibi durumlarda, normal adli kurumların yeni duruma hâkim olamayıp vazifesini yerine getiremediği veya yavaş getirdiği takdirde, askerî otoritelerin ve kurumların tercih edilmesi ile kullanıma girer.

Sıkıyönetim; tabii afet, salgın h


# 🔍 语法测试：土耳其语最小对立项

困惑度（Perplexity）可以告诉我们模型能更好地预测土耳其语文本。为了验证它确实学到了土耳其语语法，而不仅仅是维基百科的词汇，我们使用 [TurBLiMP](https://huggingface.co/datasets/juletxara/turblimp) 基准测试。这是一个由最小对立项（minimal pairs）组成的测试集：两句话仅在某一个细节上有差异，其中只有一句是符合语法的。当模型赋予符合语法的句子更高的概率时，它就“通过”了该对立项。这些对立项涵盖了 16 种语言现象（一致关系、绑定、词序等），而且它们都不来自维基百科。

我们对每种语言现象使用 100 对测试集，并在训练前对模型进行评分。

In [5]:
from datasets import concatenate_datasets

phenomena = ["anaphor_agreement", "argument_structure_ditransitive", "argument_structure_transitive", "binding", "determiners", "ellipsis", "irregular_forms", "island_effects", "nominalization", "npi_licensing", "passives", "quantifiers", "relative_clauses", "scrambling", "subject_agreement", "suspended_affixation"]
pairs = concatenate_datasets([load_dataset("juletxara/turblimp", p, split="train").shuffle(seed=0).select(range(100)) for p in phenomena])

@torch.no_grad()
def sentence_logprobs(model, sentences, batch_size=64):
    """计算每个句子的总对数概率。"""
    model.eval()
    tokenizer.padding_side = "right"
    scores = []
    for i in range(0, len(sentences), batch_size):
        inputs = tokenizer([tokenizer.bos_token + s for s in sentences[i:i + batch_size]], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        logits = model(**inputs, use_cache=False).logits.float()
        token_logprobs = torch.log_softmax(logits[:, :-1], dim=-1).gather(-1, inputs["input_ids"][:, 1:, None])[..., 0]
        scores += (token_logprobs * inputs["attention_mask"][:, 1:]).sum(-1).tolist()
    return scores

def grammar_test(model):
    """测试模型是否更倾向于符合语法的句子。"""
    good = sentence_logprobs(model, list(pairs["sentence_good"]))
    bad = sentence_logprobs(model, list(pairs["sentence_bad"]))
    return [g > b for g, b in zip(good, bad)]

def by_phenomenon(passed):
    return {p: sum(ok for ok, q in zip(passed, pairs["linguistic_phenomenon"]) if q == p) / sum(q == p for q in pairs["linguistic_phenomenon"]) for p in sorted(set(pairs["linguistic_phenomenon"]))}

original_passed = grammar_test(model)
print(f"📏 TurBLiMP accuracy before training: {sum(original_passed) / len(pairs):.1%}")

README.md:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

data/anaphor_agreement-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B / 45.3kB            

data/anaphor_agreement-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

data/argument_structure_ditransitive-000(…): reconstructing file:   0%|          |  0.00B / 41.3kB            

data/argument_structure_ditransitive-000(…): downloading bytes:           |  0.00B            

... (40 outputs left out)

data/suspended_affixation-00000-of-00001(…): reconstructing file:   0%|          |  0.00B / 39.8kB            

data/suspended_affixation-00000-of-00001(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

📏 TurBLiMP accuracy before training: 69.5%


# 🎯 持续预训练 (CPT + LoRA)

## 使用 PEFT 包装模型

对于持续预训练（CPT），我们通常使用比监督微调（SFT）更高的 LoRA Rank，因为模型需要学习更多新知识。我们对注意力机制投影层（`q_proj`, `k_proj`, `v_proj`, `o_proj`）、门控短卷积层（`in_proj`, `out_proj`）以及 MLP 层（`w1`, `w2`, `w3`）进行适配。LFM3 的注意力层还具有每个头的输出门（`gate_proj`）；将它添加到目标模块中也是一个合理的选择。

In [6]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=64,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj", "w1", "w2", "w3",
        # "gate_proj",  # 可选：LFM3 逐头注意力门
    ],
    task_type="CAUSAL_LM",
)

## 配置训练参数

由于数据集包含 `text` 列，因此 `SFTTrainer` 将在每个 token 上进行训练，这与预训练类似。我们还启用了数据打包（packing），就像在预训练中一样：`packing_strategy="wrapped"` 会将所有文章连接起来，用结束符（EOS token）分隔，并将文本流切分成长度正好为 2,048 个 token 的行。这样可以避免在填充（padding）上浪费计算资源，且长文章会被分块跨行，而不是直接截断。

In [7]:
from trl import SFTConfig, SFTTrainer

cpt_config = SFTConfig(
    output_dir="./LFM3-1.7B-CPT-tr",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # <- 每次迭代累计 8 行 2048 长度的 token
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,  # <- 浮点数表示总步数的比例
    max_length=2048,
    packing=True,
    packing_strategy="wrapped",
    dataset_text_field="text",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    per_device_eval_batch_size=2,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,  # <- 对于不支持 bfloat16 的 GPU，采用带有损失缩放（loss scaling）的 float16 混合精度
    report_to="none",
)

print("🏗️  Creating CPT trainer...")
trainer = SFTTrainer(
    model=model,
    args=cpt_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

🏗️  Creating CPT trainer...


Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

trainable params: 64,421,888 || all params: 1,793,223,424 || trainable%: 3.5925


## 在训练前评估模型

困惑度（Perplexity）是平均每个 token 损失的指数：数值越低，说明模型对保留的土耳其语文本的预测能力越好。

In [8]:
baseline = trainer.evaluate()
baseline_ood = trainer.evaluate(eval_dataset=ood_dataset, metric_key_prefix="ood")
print(f"📏 Perplexity before training: {math.exp(baseline['eval_loss']):.2f} (Wikipedia), {math.exp(baseline_ood['ood_loss']):.2f} (web pages)")

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,3.245184,0,2.995653,0.432116,0.000000


Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,4.091238,0,3.662124,0.332172,0.000000


📏 Perplexity before training: 25.67 (Wikipedia), 59.81 (web pages)


## 开始训练

In [ ]:
print("🚀 Starting continued pre-training...")
trainer.train()
print("🎉 Continued pre-training completed!")
print(f"🧮 Peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.1f} GB")

final = trainer.evaluate()
final_ood = trainer.evaluate(eval_dataset=ood_dataset, metric_key_prefix="ood")
print(f"📏 Perplexity before training: {math.exp(baseline['eval_loss']):.2f} (Wikipedia), {math.exp(baseline_ood['ood_loss']):.2f} (web pages)")
print(f"📏 Perplexity after training:  {math.exp(final['eval_loss']):.2f} (Wikipedia), {math.exp(final_ood['ood_loss']):.2f} (web pages)")

🚀 Starting continued pre-training...


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,2.582692,2.519174,2.836267,0.510017,1638400.000000


# 🔍 原始模型 vs 微调后模型

我们使用训练后的模型再次运行语法测试，并观察训练改变了模型选择的那些样本对。

In [ ]:
finetuned_passed = grammar_test(trainer.model)
original_scores, finetuned_scores = by_phenomenon(original_passed), by_phenomenon(finetuned_passed)

# Pairs the original model got wrong and the fine-tuned model gets right, in the phenomena that improved most
most_improved = sorted(phenomena, key=lambda p: finetuned_scores[p] - original_scores[p], reverse=True)[:4]
for p in most_improved:
    i = next(i for i, (o, f, q) in enumerate(zip(original_passed, finetuned_passed, pairs["linguistic_phenomenon"])) if q == p and not o and f)
    print(f"\n🧩 {p.replace('_', ' ')}: {original_scores[p]:.0%} -> {finetuned_scores[p]:.0%}")
    print(f"   ✅ grammatical:   {pairs[i]['sentence_good']}")
    print(f"   ❌ ungrammatical: {pairs[i]['sentence_bad']}")
    print(f"   Original prefers the ungrammatical one, the fine-tuned model prefers the grammatical one.")

以下是总结报告：

In [ ]:
print(f"{'':<34}{'original':>10}{'fine-tuned':>12}")
print(f"{'perplexity, Wikipedia (held out)':<34}{math.exp(baseline['eval_loss']):>10.2f}{math.exp(final['eval_loss']):>12.2f}")
print(f"{'perplexity, web pages (OOD)':<34}{math.exp(baseline_ood['ood_loss']):>10.2f}{math.exp(final_ood['ood_loss']):>12.2f}")
print(f"{'TurBLiMP grammar accuracy':<34}{sum(original_passed) / len(pairs):>10.1%}{sum(finetuned_passed) / len(pairs):>12.1%}")
for p in phenomena:
    print(f"{'  ' + p.replace('_', ' '):<34}{original_scores[p]:>10.0%}{finetuned_scores[p]:>12.0%}")

# 💾 保存并合并模型

我们保存 LoRA 适配器，然后将其合并到基础权重中。`save_pretrained` 还会将 LFM3 的建模代码复制到权重旁，这样合并后的模型在加载时也能和原始模型一样使用 `trust_remote_code=True`。

如果您想将合并后的模型推送到 Hugging Face Hub，请取消下面代码中的注释。

In [ ]:
trainer.save_model("./LFM3-1.7B-CPT-tr-lora")
print("💾 LoRA adapter saved to: ./LFM3-1.7B-CPT-tr-lora")

output_dir = "./LFM3-1.7B-CPT-tr-merged"
print("🔄 Merging LoRA weights...")
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"💾 Merged model saved to: {output_dir}")

# 若要将合并后的模型推送到 Hugging Face Hub，请取消以下注释：
# merged_model.push_to_hub("your-username/LFM3-1.7B-CPT-tr")
# tokenizer.push_to_hub("your-username/LFM3-1.7B-CPT-tr")

# 🚀 后续步骤

- **使用更多数据**：CPT 能够从比本演示多得多的文本中受益。可以增加文章数量，或者使用 `load_dataset("json", data_files=..., streaming=True)` 流式传输您自己的语料库。
- **教导模型在新的领域中遵循指令**：在 CPT 之后，可以从合并后的检查点开始，在来自该领域的对话数据上进行简短的监督微调（SFT）（参见 [`lfm3_sft_with_trl.ipynb`](./lfm3_sft_with_trl.ipynb)）。
- **同时训练词嵌入（Embeddings）**：对于包含大量未见词汇的新语言，同时训练输入嵌入层会有很大帮助。